In [1]:
# Install dependencies
%pip install torch
%pip install -U transformers==4.57.1 trl==0.25.1 datasets==4.4.1
!pip install bitsandbytes
!pip install peft

In [2]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

30

In [3]:
# Authenticate with HuggingFace
from google.colab import userdata
from huggingface_hub import login

In [4]:
# Set your HF_TOKEN in Colab secrets (key icon in sidebar)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [5]:
# Generate the training dataset
!python generate_calendar_dataset.py

Generated 1260 examples
  Train: 1117
  Eval:  143
Saved to: calendar_training_data.jsonl


In [6]:
# Load FunctionGemma-270M
from transformers import AutoTokenizer, AutoModelForCausalLM

gemma_model = "google/functiongemma-270m-it"
base_model = AutoModelForCausalLM.from_pretrained(
    gemma_model,
    device_map="auto",
    attn_implementation="eager",
    dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained(gemma_model)

In [7]:
# Load and format the training data
import json
from datasets import load_dataset

dataset = load_dataset("json", data_files="calendar_training_data.jsonl")["train"].shuffle(seed=42)

def apply_format_and_tokenize(sample):
    # 1. Format prompt + completion strings (injecting the tools!)
    prompt_messages = sample['messages'][:-1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tools=sample['tools'],
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        sample['messages'],
        tools=sample['tools'],
        tokenize=False,
        add_generation_prompt=False
    )

    # 2. Tokenize into IDs (add_special_tokens=False because the template handles BOS/EOS)
    prompt_tokens = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_tokens = tokenizer(full_text, add_special_tokens=False)

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    # 3. Create labels and mask the prompt with -100 (completion-only loss)
    labels = input_ids.copy()
    prompt_len = len(prompt_tokens)
    labels[:prompt_len] = [-100] * prompt_len

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "split": sample.get("metadata", "train"),
    }

processed_dataset = dataset.map(apply_format_and_tokenize)
train_dataset = processed_dataset.filter(lambda x: x['split'] == 'train')
eval_dataset = processed_dataset.filter(lambda x: x['split'] == 'eval')

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
max_tokens = max(len(x['input_ids']) for x in processed_dataset) + 100
print(f"Max token count: {max_tokens}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1260 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1260 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1260 [00:00<?, ? examples/s]

Train: 1117, Eval: 143
Max token count: 3456


In [8]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch

# Switch to the standard Trainer to bypass TRL's strict formatting assumptions
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

output_dir = "/content/calendar-functiongemma"

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,     # <-- FORCE EVAL TO BATCH SIZE 1
    eval_accumulation_steps=1,        # <-- OFFLOAD EVAL MEMORY TO CPU
    gradient_accumulation_steps=32,
    logging_strategy="steps",
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    save_strategy="epoch",
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=True,
    report_to="none"
)

base_model.config.pad_token_id = tokenizer.pad_token_id
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=base_model, padding=True)

base_model.enable_input_require_grads()

# 1. Define the LoRA adapter configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Wrap your base model with the adapter
base_model = get_peft_model(base_model, peft_config)
base_model.print_trainable_parameters() # This will show you are only training ~1% of the model!

trainer = Trainer(
    model=base_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
)

trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Training complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 1,474,560 || all params: 269,572,736 || trainable%: 0.5470


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
50,2.611000,1.657366
100,1.146900,1.302808


Training complete!


In [9]:
# Push to HuggingFace Hub
from huggingface_hub import whoami

trained_model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(output_dir)

username = whoami()['name']
hf_repo_id = f"{username}/functiongemma-270m-calendar-agent"

trained_model.push_to_hub(hf_repo_id, create_repo=True, commit_message="Calendar agent fine-tune")
tokenizer.push_to_hub(hf_repo_id)
print(f"Uploaded to: https://huggingface.co/{hf_repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  96%|#########5| 5.65MB / 5.92MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pskxuvkxg/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...mpskxuvkxg/tokenizer.json:  24%|##3       | 7.97MB / 33.4MB            

Uploaded to: https://huggingface.co/amoghghadge/functiongemma-270m-calendar-agent


In [1]:
# Install conversion tools
!pip uninstall -y tensorflow
!pip install ai-edge-torch-nightly --force-reinstall
!pip install ai-edge-litert-nightly huggingface_hub transformers==4.57.1 peft

Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 523.6/523.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.6/576.6 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 125.1 MB/s eta 0:

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 142.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.3.0
    Uninstalling transformers-5.3.0:
      Successfully uninstalled transformers-5.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.


In [1]:
import os
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from ai_edge_torch.generative.examples.gemma3 import gemma3
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig
from ai_edge_torch.generative.layers import kv_cache

# 1. Authenticate (Requires your HF_TOKEN in Colab secrets)
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

# 2. Define our models
base_model_id = "google/functiongemma-270m-it"
adapter_id = "amoghghadge/functiongemma-270m-calendar-agent"
merged_dir = "/content/merged-model"
os.makedirs(merged_dir, exist_ok=True)

# 3. Download and Merge the LoRA adapter into the Base Model
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

print("Merging your fine-tuned adapter...")
peft_model = PeftModel.from_pretrained(base_model, adapter_id)
merged_model = peft_model.merge_and_unload()

print("Saving full merged model locally...")
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

# 4. Setup LiteRT Metadata
llm_metadata = r"""start_token: { token_ids: { ids: [ 2 ] } }
stop_tokens: { token_str: "<end_of_turn>" }
stop_tokens: { token_str: "<start_function_response>" }
llm_model_type: { function_gemma: {} }
"""

litertlm_dir = '/content/litertlm'
os.makedirs(litertlm_dir, exist_ok=True)
metadata_path = os.path.join(litertlm_dir, 'base_llm_metadata.textproto')
with open(metadata_path, 'w') as f:
    f.write(llm_metadata)

# 5. Load the newly merged PyTorch Model into AI Edge
print("\nBuilding PyTorch model for LiteRT...")
pytorch_model = gemma3.build_model_270m(merged_dir) # We now point to the merged model!

# 6. Setup Export Configurations
export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

# 7. Convert to LiteRT
print("Starting LiteRT conversion... this might take a few minutes.")
converter.convert_to_litert(
    pytorch_model,
    output_path=litertlm_dir,
    output_name_prefix="calendar-agent",
    prefill_seq_len=256,
    kv_cache_max_len=4096,
    quantize="dynamic_int8",
    export_config=export_config,
    tokenizer_model_path=os.path.join(merged_dir, 'tokenizer.model'),
    base_llm_metadata_path=metadata_path,
    output_format="litertlm",
)
print("\n🎉 Conversion complete! The model is ready for Xcode.")

/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:91: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.9.1, so it will not be used.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_xla2/distributed.py:106: UserWarning: Device capability of jax unspecified, assuming `cpu` and `cuda` or `xpu`. Please specify it via the `devices` argument of `register_backend`.
  dist.Backend.register_backend("jax", ProcessGroupJax)
ERROR:jax._src.xla_bridge:Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 497, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda12/__init__.py", line 348, in initialize
    xla_client.register_custom_type_id_handler(
    ^^^^^^^^^^^^^^^^^

Loading base model...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/5.92M [00:00<?, ?B/s]

Saving full merged model locally...

Building PyTorch model for LiteRT...
Starting LiteRT conversion... this might take a few minutes.


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/usr/local/lib/python3.12/dist-packages/ai_edge_torch/_convert/signature.py:52: FutureWarning: `treespec.children_specs` is deprecated. Use `treespec.child(index)` to access a single child, or `treesp


🎉 Conversion complete! The model is ready for Xcode.


In [2]:
# Save to Google Drive for easy download
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/edgebridge-models/
!cp /content/litertlm/calendar-agent_q8_ekv4096.litertlm /content/drive/MyDrive/edgebridge-models/

print("Model saved to Google Drive! You are ready for Xcode.")

Mounted at /content/drive
Model saved to Google Drive! You are ready for Xcode.
